# Project 27 — Freshness-Aware News RAG
## Prioritize Recent Documents in Retrieval

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup with Timestamped Articles

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from datetime import datetime, timedelta

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Simulated news articles at different dates
base_date = datetime(2025, 1, 15)
articles = [
    Document(page_content="AI startup raises $50M Series B to build enterprise RAG solutions. "
        "The company plans to hire 100 engineers.", metadata={
        "date": (base_date - timedelta(days=1)).isoformat(), "days_ago": 1, "source": "techcrunch"}),
    Document(page_content="New open-source embedding model achieves state-of-the-art on MTEB "
        "benchmark. Available on HuggingFace.", metadata={
        "date": (base_date - timedelta(days=7)).isoformat(), "days_ago": 7, "source": "arxiv"}),
    Document(page_content="AI startup raises $20M Series A for enterprise RAG platform. "
        "Focuses on healthcare and legal verticals.", metadata={
        "date": (base_date - timedelta(days=90)).isoformat(), "days_ago": 90, "source": "techcrunch"}),
    Document(page_content="Ollama releases version 0.5 with multi-model support and improved "
        "performance. Memory usage reduced by 30%.", metadata={
        "date": (base_date - timedelta(days=3)).isoformat(), "days_ago": 3, "source": "github"}),
    Document(page_content="Study shows RAG systems reduce hallucination by 70% compared to "
        "base LLMs in enterprise settings.", metadata={
        "date": (base_date - timedelta(days=30)).isoformat(), "days_ago": 30, "source": "research"}),
]

vectorstore = Chroma.from_documents(articles, embeddings, collection_name="news_rag")
print(f"Indexed {len(articles)} articles with timestamps")

## Step 2 — Standard Retrieval (Ignores Freshness)

In [ ]:
query = "What's the latest news about AI startups and RAG?"
results = vectorstore.similarity_search_with_score(query, k=3)

print("Standard retrieval (no freshness weighting):")
for doc, score in results:
    print(f"  [{doc.metadata['days_ago']}d ago] score={score:.4f} — {doc.page_content[:60]}...")

## Step 3 — Freshness-Weighted Retrieval

In [ ]:
import math

def freshness_search(query, k=3, freshness_weight=0.3, max_days=180):
    """Combine semantic similarity with time recency."""
    results = vectorstore.similarity_search_with_score(query, k=len(articles))
    scored = []
    for doc, sim_score in results:
        days_ago = doc.metadata.get("days_ago", max_days)
        # Freshness: exponential decay
        freshness = math.exp(-days_ago / 30.0)
        # Combined score (lower sim_score = better in Chroma)
        combined = (1 - freshness_weight) * (1 / (1 + sim_score)) + freshness_weight * freshness
        scored.append((doc, combined, sim_score, freshness))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]

print("Freshness-weighted retrieval:")
for doc, combined, sim, fresh in freshness_search(query):
    print(f"  [{doc.metadata['days_ago']}d ago] combined={combined:.4f} "
          f"sim={sim:.4f} fresh={fresh:.4f} — {doc.page_content[:60]}...")

## Step 4 — Compare Results

In [ ]:
queries = [
    "Latest AI startup funding news",
    "What's new with Ollama?",
    "RAG hallucination research",
]

for q in queries:
    print(f"\nQ: {q}")
    standard = vectorstore.similarity_search(q, k=2)
    fresh = freshness_search(q, k=2)

    std_ids = [(d.metadata['days_ago'], d.metadata['source']) for d in standard]
    fresh_ids = [(d.metadata['days_ago'], d.metadata['source']) for d, _, _, _ in fresh]

    print(f"  Standard: {std_ids}")
    print(f"  Fresh:    {fresh_ids}")

## What You Learned
- **Time-aware retrieval** using exponential freshness decay
- **Score combination** blending semantic similarity with recency
- **Configurable freshness weight** for different use cases